# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-02_tokens_and_transformer.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day1/T1-02_tokens_and_transformer.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day1"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day1" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


Token counts use the classroom stand-in tokenizer (no Hugging Face download). The decoding cell calls `lab.chat` in demo mode; temperature is ignored — the teaching note still holds for live vLLM.


# T1-02 Tokens and the transformer

**Time:** 10:45–12:00 &nbsp;|&nbsp; **Evidence:** token checkpoint (`logs/T1-02_tokens.json`) + marked architecture card

**Notebook learning outcomes**

- See OLMo's own token pieces and IDs for a sentence, and decode them back to the same text.
- Find where the same token ID appears in two different sentences, and say what still differs inside the model.
- Count a full chat request (system + user + role markers) against its content, and relate it to the context window.
- Run the same prompt at two temperatures, and say what changed, what didn't, and what you can't conclude.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Ministry link

The sentences below are **fictional**. They are shaped like the prose each cluster actually lives with. You are counting OLMo's own tokens, not retrieving classified files.

| Sentence | Stands in for |
|---|---|
| A | Finance / RDLG — minute-style prose |
| B | Defence OPR / Labour — a numbered rule |
| C | Trade — a ticket in ordinary English |
| D | Defence assets / MOWI — a register-shaped question |

**N12 woven:** the **weights** you called in T1-01 *are* this machine (embedding + blocks + head), compressed from training data you will look at as a licence question in T1-04.

## Overnight readers

T1-01 sent a string and got a string. This notebook opens the pipe in the middle. The model never sees letters. It sees **token IDs**. A long Finance minute pack or 2,000 Labour applications can blow the **context window**. That is why Day 2 retrieves *chunks*, not whole systems.


## Architecture card (keep this sheet for Day 3)

```
input text
    → tokenizer          (text to token IDs)
    → embedding          (IDs to vectors)
    → transformer block × N     (attention, then feed-forward)
    → language-model head       (vector to next-token scores)
    → chosen token appended, repeat until stop
```

**Training** updates weights (or a small adapter). **Inference** uses frozen weights to pick the next token. Today is inference only.

On Day 3 you will shade what LoRA updates. Today, shade nothing — everything is frozen.

Today you touch the first box (the tokenizer) directly on this machine. The rest you reach through the endpoint. Nothing is shaded today.


In [ ]:
# [RUN] load OLMo's own tokenizer from the course image. No weights, no download.
import civicdesk_lab as lab

try:
    info = lab.tokenizer_info()
    for k in ("source", "backend", "vocab_size", "context_window", "tokenizer_sha256", "weights_loaded"):
        print(f"{k:18} {info[k]}")
except lab.TokenizerUnavailable as exc:
    print(exc)


The tokenizer is a fixed list of text pieces, about the size printed above as `vocab_size`, and it was settled before OLMo was trained. The same text always gives the same IDs. This step is not the model thinking: it is a lookup that happens before the first block runs. In the next cell, a `␠` means the piece starts with a space.


In [ ]:
# [RUN] the two unfinished sentences from this morning, as OLMo sees them
bank_money = "I need to deposit my check, so I’m going to the bank to"
bank_river = "I’ve got my fishing rod, so I’m heading to the river bank to"

for label, text in [("money", bank_money), ("river", bank_river)]:
    print(f"== {label}: {text!r}   tokens={lab.count_tokens(text)}")
    lab.show_tokens(text)
    rt = lab.roundtrip(text)
    print("decoded back:", repr(rt["decoded"]), "| identical to input:", rt["identical"])
    print()


In [ ]:
# [RUN] find "bank" in each sentence. Read what this tokenizer actually did.
found = {}
for label, text in [("money", bank_money), ("river", bank_river)]:
    rows = lab.find_token(text, "bank")
    found[label] = rows
    print(label, [(r["pos"], r["id"], repr(r["text"])) for r in rows])

ids_money = [r["id"] for r in found["money"]]
ids_river = [r["id"] for r in found["river"]]
if ids_money == ids_river:
    print("Observed: the same ID(s) for 'bank' in both sentences:", ids_money)
else:
    print("Observed: different IDs here:", ids_money, "vs", ids_river, "- check spacing and case.")

print("\nSame letters, small changes (observed, this tokenizer only):")
for s in ["bank", " bank", "Bank", " Bank", "banks"]:
    print(f"  {s!r:9} -> {lab.token_ids(s)}")


In these two sentences, check whether "bank" got the same ID. It does when the piece and its leading space match. Change the space or the capital letter and the ID changes, which is why we compare matching strings.

The same ID means the same **starting** vector: the embedding step is a table lookup. On the way through the transformer blocks, each position mixes in the other tokens before it ("deposit… check" in one sentence, "fishing rod… river" in the other). By the time the head scores the next token, the vectors at "bank" are no longer the same. That's why the two sentences can be continued differently. We don't measure those inner vectors today, because that needs the weights loaded in this notebook. See the architecture card: the tokenizer and embedding are the same for both sentences, and the blocks are where they part.

On your card, write your own guess for the next word in each sentence.


In [ ]:
# [RUN] the four fictional ministry sentences, counted with OLMo's tokenizer
sentences = {
    "A": "The committee noted the request dated 12 March and asked that the attachment be filed with the minute.",
    "B": "Criterion C2: the applicant must hold a recognised diploma in records management or equivalent.",
    "C": "Printer on the third floor will not take paper. User already restarted. Please advise.",
    "D": "List CivicDesk laptops assigned to Unit 4 whose warranty ends this month.",
}
for k, s in sentences.items():
    rows = lab.tokenize(s)
    pieces = [r["text"] for r in rows]
    print(f"{k}  words={len(s.split()):3d}  tokens={len(rows):3d}  {pieces[:12]}{'...' if len(pieces) > 12 else ''}")


These are OLMo's real pieces now, not a stand-in. Notice that tokens and words aren't the same count, and that the gap depends on the kind of text: codes like "C2", dates and numbers split differently from ordinary words.

The **context window** printed in the first cell is a budget in tokens, not words. The system rules, the user turn, any retrieved chunks and the reply all share it. A long Cabinet pack (Finance) or a 2,000-application batch (Labour) can overrun it even when a person could skim it. That's one reason Day 2 retrieves *chunks*, not whole EDMS filings.

`# [CHANGE]` below: take sentence B or C and add one word. Watch the count move. Then try `I'm` against `I’m`, or a place name.


In [ ]:
# [CHANGE] edit this string, re-run, write the new count on the architecture card
probe = "Criterion C2: the applicant must hold a recognised diploma in records management or equivalent."
print("tokens:", lab.count_tokens(probe), "| decodes back identically:", lab.roundtrip(probe)["identical"])
lab.show_tokens(probe)


## What the model actually receives

In T1-01 you sent a `system` message and a `user` message. The model doesn't receive two neat boxes. The runtime pastes them into one string using OLMo's **chat template**: a start marker, a `<|system|>` label, your rules, a `<|user|>` label, your text, and finally `<|assistant|>` to say "your turn" (the **generation prompt**). Then it tokenizes that whole string. The next cell shows the string and counts it.

The system text below is a scope rule: finance questions only. It is only more tokens in the window. It changes **no weights**, and the model may still not follow it, so it has to be tested, not assumed.


In [ ]:
# [RUN] count the full chat request, then the content alone
FINANCE_ONLY = (
    "CivicDesk finance assistant. Answer only questions about CivicDesk finance procedures: "
    "budgets, payment vouchers, purchase orders and expense claims. "
    "If a request is about anything else, reply exactly: Out of scope: finance questions only."
)
cases = {
    "finance rule + money sentence": [{"role": "system", "content": FINANCE_ONLY}, {"role": "user", "content": bank_money}],
    "finance rule + river sentence": [{"role": "system", "content": FINANCE_ONLY}, {"role": "user", "content": bank_river}],
    "money sentence, no system":     [{"role": "user", "content": bank_money}],
}
acct = {}
for name, msgs in cases.items():
    a = lab.chat_tokens(msgs)
    acct[name] = a
    parts = " + ".join(f"{m['role']} {m['content_tokens']}" for m in a["per_message"])
    print(f"{name:32} full={a['count']:3d}  content={parts} = {a['content_sum']}  difference={a['difference']}")

first = acct["finance rule + money sentence"]
print("\nThe string the model receives (first case):")
print(repr(first["rendered"]))
print("\nContent tokens still in one unbroken run inside the full request?")
for m in first["per_message"]:
    print(f"  {m['role']:6} {m['appears_unbroken']}")

window = lab.tokenizer_info()["context_window"]
REPLY_BUDGET = 256  # what lab.chat asks for by default
if window:
    print(f"\nWindow {window} - request {first['count']} - reply budget {REPLY_BUDGET} = "
          f"{window - first['count'] - REPLY_BUDGET} tokens left for retrieved chunks (Day 2).")


The full request is longer than the content: the start marker, role labels, newlines and the generation prompt all cost tokens. Compare the differences across the three cases. They aren't one fixed number per message, and where your text meets the template, the tokenizer can merge characters differently than it does when the text stands alone. For example, a final full stop can merge with the newline after it. The "unbroken run" check shows where this happens. So measure with the real template. Don't estimate a request by adding a constant to your word count.

The scope rule costs those tokens on every call, and it doesn't change OLMo. The next cell tests whether this model follows it on this backend.


In [ ]:
# [RUN] does OLMo keep to the finance-only rule? Two questions, temperature 0, short replies.
scope_tests = [
    ("in scope",     "What should be attached to a CivicDesk payment voucher before approval?"),
    ("out of scope", bank_river),
]
scope_runs = []
for label, q in scope_tests:
    r = lab.chat([{"role": "system", "content": FINANCE_ONLY}, {"role": "user", "content": q}],
                 temperature=0.0, max_tokens=60)
    scope_runs.append({"label": label, "question": q, "mode": r["mode"], "engine": r["engine"], "text": r["text"]})
    print(f"[{label}] mode={r['mode']} engine={r['engine']}")
    print("   ", r["text"][:300])
if any(x["mode"] != "live" for x in scope_runs):
    print("\nDEMO / FALLBACK: canned text. This shows nothing about whether OLMo follows the rule.")
else:
    print("\nJudge it yourself: did the out-of-scope reply say exactly the refusal line? One test is not a guarantee.")


At every step the head gives a score to every possible next token. **Temperature** rescales those scores before one token is picked. At 0 the top-scoring token is always taken (greedy). Higher values flatten the choice, so lower-ranked tokens get picked more often. It acts only at inference: OLMo's weights are the same at every setting.

Low temperature makes a reply **more repeatable**, not more correct. A wrong answer can be repeated perfectly. The next cell prints which backend you're on, and whether the server reports applying the setting.


In [ ]:
# [RUN] same CivicDesk prompt, two temperatures, repeated samples
cfg = lab.endpoint_config()
print("backend:", cfg["backend"], "| engine:", cfg.get("engine"), "| model:", cfg["model"])

messages = [
    {"role": "system", "content": "CivicDesk classroom. One short sentence. No invented citations."},
    {"role": "user", "content": "In one sentence: should a missing diploma be Meets, Does not meet, or Human review required?"},
]
SETTINGS = [(0.0, 2), (0.9, 3)]  # [CHANGE] (temperature, how many samples)
runs = []
for temp, repeats in SETTINGS:
    for k in range(repeats):
        r = lab.chat(messages, temperature=temp, max_tokens=60)
        runs.append(r)
        print(f"requested={r['requested']['temperature']}  applied={r['applied'].get('temperature')}  "
              f"mode={r['mode']}  engine={r['engine']}  sample {k + 1}")
        print("   ", r["text"][:240])

if any(r["canned"] for r in runs):
    print("\nDEMO / FALLBACK: canned replies. Temperature was not applied, so identical text tells you nothing.")
else:
    for temp, _ in SETTINGS:
        texts = [r["text"] for r in runs if r["requested"]["temperature"] == temp]
        print(f"temperature {temp}: {len(set(texts))} different repl{'y' if len(set(texts)) == 1 else 'ies'} out of {len(texts)}")
    print("Write down what you saw. A few samples can match or differ at either setting; neither proves the answer is right.")


### Teaching note — decoding

Temperature is not "creativity." It is how sharply the head picks among next tokens. For **Labour / Health / OPR** criteria work you want the boring setting (low temperature) plus a quote. For a Trade ticket draft you might tolerate more variation, but it's still not a password reset. A repeatable answer still needs its quote checked by a person.

**Training vs inference:** nobody in this room is updating OLMo's blocks today. The scope rule and the temperature are both settings on a call. Neither changes a weight.


In [ ]:
# [RUN] save the evidence. This is the token checkpoint.
log_path = lab.save_log("T1-02_tokens", {
    "tokenizer": lab.tokenizer_info(),
    "bank": {k: [{x: r[x] for x in ("pos", "id", "text")} for r in v] for k, v in found.items()},
    "probe": {"text": probe, "tokens": lab.count_tokens(probe)},
    "chat_accounting": {k: {x: v[x] for x in ("count", "content_sum", "difference")} for k, v in acct.items()},
    "scope_test": scope_runs,
    "temperature_runs": [{x: r.get(x) for x in ("mode", "engine", "canned", "requested", "applied", "usage", "text")} for r in runs],
    "backend": lab.endpoint_config(),
})
print("saved", log_path)


## Write-up

1. Mark the architecture card: tokenizer, embedding, blocks, head.
2. Did " bank" get the same ID in both sentences? Even so, what differs inside the model by the time it picks the next word?
3. Your finance-rule request: full count, content count, difference. Why can't you estimate it by adding a constant?
4. Using the context-window number, why might Finance *not* paste a whole minute pack into this window?
5. What did temperature change on **your** backend? What was `mode`? If it was `demo`, what can you not conclude?
6. One sentence: what is frozen right now? Did the scope rule change any of it?


In [ ]:
# [OPTIONAL] how big are Day 2's handbook chunks, in OLMo tokens?
import json
from statistics import median
chunks = json.loads((lab.HERE.parent / "day2" / "data" / "handbooks.json").read_text(encoding="utf-8"))["chunks"]
sizes = sorted(lab.count_tokens(c["text"]) for c in chunks if c.get("index_ok"))
print(f"{len(sizes)} indexable chunks: min {sizes[0]}, median {median(sizes)}, max {sizes[-1]} tokens")


In [ ]:
# [OPTIONAL] send each bank sentence to the OLMo already running. One short request each.
for label, text in [("money", bank_money), ("river", bank_river)]:
    r = lab.chat([{"role": "user", "content": text}], temperature=0.0, max_tokens=12)
    print(f"{label}: mode={r['mode']} -> {r['text']!r}")


This is an Instruct model behind a chat template, so it may answer like an assistant rather than finish your sentence. That's the chat format at work.
